<a href="https://colab.research.google.com/github/taywhipp/ASB-ALA-Catalog-Marketing/blob/main/data_cleaning_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NHL Career Longevity, Data Cleaning and Feature Engineering

SIADS 696 Milestone II  
Team: Taylor Whipple, Zain Adhami, Meagan McBride

This notebook builds the modeling table. We combine MoneyPuck season stats, NHL API bios, the injury database, and CapWages contract data into one row per player-season, then engineer the features and target we need for the longevity models.

## Setup

We import everything up front, mount Drive, and set the project path.

In [ ]:
# Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data sources and parsing
import requests
import json
import re
import os
import time
import html as ihtml
import unicodedata
from tqdm.auto import tqdm

# Modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
# Mount the Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

path = "/content/drive/MyDrive/SIADS 696: Milestone II/Money_Puck_Yearly_Data"

## Step 1. Load and combine MoneyPuck files

We read the per-season MoneyPuck skater files and concatenate them, then keep only the rows where situation equals "all" so each player has one combined row per season.

In [ ]:
files = [
    "skaters_2008_to_2024.csv",
    "skaters_2025_2026.csv",
]

season_files = [pd.read_csv(f"{path}/{f}") for f in files]
df_combined = pd.concat(season_files, ignore_index=True)
df_combined.head()

,playerId,season,name,team,position,situation,games_played,icetime,shifts,gameScore,onIce_xGoalsPercentage,offIce_xGoalsPercentage,onIce_corsiPercentage,offIce_corsiPercentage,onIce_fenwickPercentage,offIce_fenwickPercentage,iceTimeRank,I_F_xOnGoal,I_F_xGoals,I_F_xRebounds,I_F_xFreeze,I_F_xPlayStopped,I_F_xPlayContinuedInZone,I_F_xPlayContinuedOutsideZone,I_F_flurryAdjustedxGoals,I_F_scoreVenueAdjustedxGoals,I_F_flurryScoreVenueAdjustedxGoals,I_F_primaryAssists,I_F_secondaryAssists,I_F_shotsOnGoal,I_F_missedShots,I_F_blockedShotAttempts,I_F_shotAttempts,I_F_points,I_F_goals,I_F_rebounds,I_F_reboundGoals,I_F_freeze,I_F_playStopped,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,penalties,I_F_penalityMinutes,I_F_faceOffsWon,I_F_hits,I_F_takeaways,I_F_giveaways,I_F_lowDangerShots,I_F_mediumDangerShots,I_F_highDangerShots,I_F_lowDangerxGoals,I_F_mediumDangerxGoals,I_F_highDangerxGoals,I_F_lowDangerGoals,I_F_mediumDangerGoals,I_F_highDangerGoals,I_F_scoreAdjustedShotsAttempts,I_F_unblockedShotAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_dZoneGiveaways,I_F_xGoalsFromxReboundsOfShots,I_F_xGoalsFromActualReboundsOfShots,I_F_reboundxGoals,I_F_xGoals_with_earned_rebounds,I_F_xGoals_with_earned_rebounds_scoreAdjusted,I_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted,I_F_shifts,I_F_oZoneShiftStarts,I_F_dZoneShiftStarts,I_F_neutralZoneShiftStarts,I_F_flyShiftStarts,I_F_oZoneShiftEnds,I_F_dZoneShiftEnds,I_F_neutralZoneShiftEnds,I_F_flyShiftEnds,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,OnIce_F_xOnGoal,OnIce_F_xGoals,OnIce_F_flurryAdjustedxGoals,OnIce_F_scoreVenueAdjustedxGoals,OnIce_F_flurryScoreVenueAdjustedxGoals,OnIce_F_shotsOnGoal,OnIce_F_missedShots,OnIce_F_blockedShotAttempts,OnIce_F_shotAttempts,OnIce_F_goals,OnIce_F_rebounds,OnIce_F_reboundGoals,OnIce_F_lowDangerShots,OnIce_F_mediumDangerShots,OnIce_F_highDangerShots,OnIce_F_lowDangerxGoals,OnIce_F_mediumDangerxGoals,OnIce_F_highDangerxGoals,OnIce_F_lowDangerGoals,OnIce_F_mediumDangerGoals,OnIce_F_highDangerGoals,OnIce_F_scoreAdjustedShotsAttempts,OnIce_F_unblockedShotAttempts,OnIce_F_scoreAdjustedUnblockedShotAttempts,OnIce_F_xGoalsFromxReboundsOfShots,OnIce_F_xGoalsFromActualReboundsOfShots,OnIce_F_reboundxGoals,OnIce_F_xGoals_with_earned_rebounds,OnIce_F_xGoals_with_earned_rebounds_scoreAdjusted,OnIce_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted,OnIce_A_xOnGoal,OnIce_A_xGoals,OnIce_A_flurryAdjustedxGoals,OnIce_A_scoreVenueAdjustedxGoals,OnIce_A_flurryScoreVenueAdjustedxGoals,OnIce_A_shotsOnGoal,OnIce_A_missedShots,OnIce_A_blockedShotAttempts,OnIce_A_shotAttempts,OnIce_A_goals,OnIce_A_rebounds,OnIce_A_reboundGoals,OnIce_A_lowDangerShots,OnIce_A_mediumDangerShots,OnIce_A_highDangerShots,OnIce_A_lowDangerxGoals,OnIce_A_mediumDangerxGoals,OnIce_A_highDangerxGoals,OnIce_A_lowDangerGoals,OnIce_A_mediumDangerGoals,OnIce_A_highDangerGoals,OnIce_A_scoreAdjustedShotsAttempts,OnIce_A_unblockedShotAttempts,OnIce_A_scoreAdjustedUnblockedShotAttempts,OnIce_A_xGoalsFromxReboundsOfShots,OnIce_A_xGoalsFromActualReboundsOfShots,OnIce_A_reboundxGoals,OnIce_A_xGoals_with_earned_rebounds,OnIce_A_xGoals_with_earned_rebounds_scoreAdjusted,OnIce_A_xGoals_with_earned_rebounds_scoreFlurryAdjusted,OffIce_F_xGoals,OffIce_A_xGoals,OffIce_F_shotAttempts,OffIce_A_shotAttempts,xGoalsForAfterShifts,xGoalsAgainstAfterShifts,corsiForAfterShifts,corsiAgainstAfterShifts,fenwickForAfterShifts,fenwickAgainstAfterShifts
0,8462196,2008,Stephane Robidas,DAL,D,other,72,6729.0,105.0,31.93,0.51,0.36,0.59,0.46,0.58,0.46,134.0,22.30,1.91,1.59,4.82,0.81,14.80,8.20,1.80,1.91,1.80,3.0,1.0,24.0,10.0,16.0,50.0,4.0,0.0,1.0,0.0,3.0,2.0,18.0,10.0,24.0,34.0,3.0,6.0,0.0,9.0,2.0,3.0,28.0,4.0,2.0,0.92,0.46,0.53,0.0,0.0,0.0,50.00,34.0,34.00,0.0,0.49,0.13,0.07,2.27,2.27,2.14,105.0,26.0,25.0,25.0,29.0,7.0,10.0,20.0,68.0,0.0,0.0,5938.0,6.0,0.0,0.0,3.0,84.29,13.21,12.22,13.21,12.22,92.0,38.0,38.0,168.0,11.

In [ ]:
df_combined_all = df_combined[df_combined["situation"] == "all"].copy()

## Step 2. Pull player bios from the NHL API

We hit the NHL API landing endpoint for every unique playerId to get birth date, size, handedness, and draft details. The pull is slow, so we cache it to player_bios.csv and only re-pull when the cache is missing.

In [ ]:
bios_cache = f"{path}/player_bios.csv"

if os.path.exists(bios_cache) and os.path.getsize(bios_cache) > 1:
    player_bios = pd.read_csv(bios_cache)
    print(f"Loaded {len(player_bios)} cached bios")
else:
    unique_ids = df_combined_all['playerId'].unique()
    print(f"Pulling bios for {len(unique_ids)} unique players")

    bio_rows, failed_ids = [], []
    for i, pid in enumerate(unique_ids, start=1):
        url = f"https://api-web.nhle.com/v1/player/{pid}/landing"
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            d = r.json()
            draft = d.get("draftDetails") or {}  # undrafted players have no draftDetails
            bio_rows.append({
                "playerId": d.get("playerId"),
                "firstName": d.get("firstName", {}).get("default"),
                "lastName": d.get("lastName", {}).get("default"),
                "birthDate": d.get("birthDate"),
                "heightInInches": d.get("heightInInches"),
                "weightInPounds": d.get("weightInPounds"),
                "shootsCatches": d.get("shootsCatches"),
                "position": d.get("position"),
                "isActive": d.get("isActive"),
                "draftYear": draft.get("year"),
                "draftRound": draft.get("round"),
                "draftOverallPick": draft.get("overallPick"),
            })
        except Exception as e:
            failed_ids.append((pid, str(e)))
        if i % 100 == 0:
            print(f"{i}/{len(unique_ids)} done ({len(failed_ids)} failed so far)")
        time.sleep(0.5)

    player_bios = pd.DataFrame(bio_rows)
    print(f"\nPulled {len(player_bios)} bios, {len(failed_ids)} failures")
    player_bios.to_csv(bios_cache, index=False)

Loaded 3058 cached bios


## Step 3. Clean the injury database

We load the injury database, drop playoff rows, keep 2008 onward, and remove columns we don't use. The records store names as "Last, First" and seasons as "2023/24", so we reshape both to line up with MoneyPuck.

In [ ]:
# Load the injury database and keep regular-season rows from 2008 onward
player_injuries = pd.read_csv("/content/drive/MyDrive/SIADS 696: Milestone II/NHL_Injury_Database_Data.csv")

injuries_clean = player_injuries[~player_injuries['Season'].str.contains('playoffs', na=False)].copy()
injuries_clean['season'] = injuries_clean['Season'].str[:4].astype(int)
injuries_clean = injuries_clean[injuries_clean['season'] >= 2008]
injuries_clean = injuries_clean.drop(columns=['Season', 'Cap Hit', 'Chip', 'Games Missed.1'])

injuries_clean.head()

,Team,Position,Player,Injury Type,Games Missed,season
6832,Anaheim,F,"Carter, Ryan",Flu,4,2008
6833,Anaheim,F,"Getzlaf, Ryan",Mouth,1,2008
6834,Anaheim,F,"Larsen, Brad",Hernia,82,2008
6835,Anaheim,F,"Marchant, Todd",Finger,10,2008
6836,Anaheim,F,"May, Brad",Knee,1,2008


### Step 3a. Match injury names to playerIds

We resolve names in passes. First an exact normalized match, then last name plus first initial plus season, then team disambiguation for the remaining collisions, and finally a small manual map for transliteration and nickname cases. Rows that never resolve are retirees absent from MoneyPuck, so we drop them.

In [ ]:
def normalize_name(name):
    # Strip accents, lowercase, trim. E.g. ' Joni (Pitkanen)' -> 'pitkanen'
    if pd.isna(name):
        return None
    s = unicodedata.normalize('NFKD', str(name))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return s.strip().lower()

def flip_name(name):
    # "Last, First" -> "First Last"
    if pd.isna(name) or ', ' not in str(name):
        return name
    last, first = str(name).split(', ', 1)
    return f"{first} {last}"

# Normalized name -> playerId lookup from MoneyPuck
name_lookup = df_combined_all[['name', 'playerId']].drop_duplicates().copy()
name_lookup['name_norm'] = name_lookup['name'].apply(normalize_name)

# Flag names that map to multiple playerIds (e.g. two Sebastian Aho's) and handle them separately
ambiguous = name_lookup.groupby('name_norm')['playerId'].nunique()
ambiguous = ambiguous[ambiguous > 1]
print(f"Ambiguous names in df_combined_all: {len(ambiguous)}")
if len(ambiguous) > 0:
    print(name_lookup[name_lookup['name_norm'].isin(ambiguous.index)].sort_values('name_norm'))

lookup_dict = (
    name_lookup[~name_lookup['name_norm'].isin(ambiguous.index)]
    .drop_duplicates(subset='name_norm')
    .set_index('name_norm')['playerId']
    .to_dict()
)

# Pass 1: flip + normalize injury names, then map to playerId
injuries_clean['name_flipped'] = injuries_clean['Player'].apply(flip_name)
injuries_clean['name_norm'] = injuries_clean['name_flipped'].apply(normalize_name)
injuries_clean['playerId'] = injuries_clean['name_norm'].map(lookup_dict)

matched = injuries_clean['playerId'].notna().sum()
unmatched = injuries_clean['playerId'].isna().sum()
print(f"\nMatched: {matched} ({matched / len(injuries_clean) * 100:.1f}%)")
print(f"Unmatched: {unmatched}")

Ambiguous names in df_combined_all: 9
                   name  playerId         name_norm
451    Alexandre Picard   8471221  alexandre picard
951    Alexandre Picard   8470678  alexandre picard
1371        Colin White   8465005       colin white
38081       Colin White   8478400       colin white
74251  Elias Pettersson   8483678  elias pettersson
48471  Elias Pettersson   8480012  elias pettersson
12136   Erik Gustafsson   8475640   erik gustafsson
34571   Erik Gustafsson   8476979   erik gustafsson
56426    Mikko Lehtonen   8482247    mikko lehtonen
106      Mikko Lehtonen   8471747    mikko lehtonen
5461       Nathan Smith   8468503      nathan smith
58466      Nathan Smith   8480979      nathan smith
946        Ryan Johnson   8460527      ryan johnson
69956      Ryan Johnson   8481564      ryan johnson
1076       Sean Collins   8473913      sean collins
19241      Sean Collins   8474744      sean collins
35401     Sebastian Aho   8478427     sebastian aho
42621     Sebastian Aho   

In [ ]:
# Pass 2: drop goalies, then match remaining rows on last name + first initial + season
goalie_mask = injuries_clean['Position'].str.startswith('G', na=False)
print(f"Dropping {goalie_mask.sum()} goalie rows")
injuries_clean = injuries_clean[~goalie_mask].copy()
print(f"Remaining: {injuries_clean.shape[0]} rows")

player_seasons = (
    df_combined_all[['playerId', 'season', 'team']]
    .drop_duplicates(subset=['playerId', 'season'])
    .merge(player_bios[['playerId', 'firstName', 'lastName']], on='playerId', how='inner')
)
player_seasons['last_norm'] = player_seasons['lastName'].apply(normalize_name)
player_seasons['first_initial'] = player_seasons['firstName'].astype(str).str.lower().str[0]

# Parse last/first from injury names, handling "(F)" annotations and initials with periods
def parse_last_first(name):
    if pd.isna(name) or ', ' not in str(name):
        return None, None
    last_part, first_part = str(name).split(', ', 1)
    last = last_part.split(' (')[0].strip().rstrip('.')
    first = first_part.split(' (')[0].strip().rstrip('.').replace('.', '')
    return last, first

unmatched_mask = injuries_clean['playerId'].isna()
parsed = injuries_clean.loc[unmatched_mask, 'Player'].apply(lambda n: pd.Series(parse_last_first(n)))
injuries_clean.loc[unmatched_mask, 'last_norm'] = parsed[0].apply(normalize_name)
injuries_clean.loc[unmatched_mask, 'first_initial_inj'] = parsed[1].str.lower().str[0]

unmatched_df = injuries_clean.loc[unmatched_mask].reset_index()
candidates = unmatched_df.merge(
    player_seasons[['last_norm', 'first_initial', 'season', 'playerId', 'team']],
    left_on=['last_norm', 'first_initial_inj', 'season'],
    right_on=['last_norm', 'first_initial', 'season'],
    how='left',
    suffixes=('_inj', '_mp'),
)

n_cands = candidates.groupby('index')['playerId_mp'].apply(lambda x: x.notna().sum())
print(f"\nResolved to single candidate: {(n_cands == 1).sum()}")
print(f"Multiple candidates (need team): {(n_cands > 1).sum()}")
print(f"No matches: {(n_cands == 0).sum()}")

Dropping 1353 goalie rows
Remaining: 15049 rows

Resolved to single candidate: 214
Multiple candidates (need team): 32
No matches: 32


In [ ]:
# Pass 3: attach single-candidate matches, then disambiguate the rest by team
single_idx = n_cands[n_cands == 1].index
single_lookup = (
    candidates[candidates['index'].isin(single_idx) & candidates['playerId_mp'].notna()]
    .set_index('index')['playerId_mp']
)
injuries_clean['playerId'].update(single_lookup)

# Build a team-name -> abbreviation mapping from already-matched rows (e.g. "Ottawa" -> "OTT")
matched_so_far = injuries_clean[injuries_clean['playerId'].notna()]
team_pairs = matched_so_far.merge(
    df_combined_all[['playerId', 'season', 'team']].rename(columns={'team': 'team_mp'}),
    on=['playerId', 'season'],
    how='inner'
)
team_mapping = team_pairs.groupby('Team')['team_mp'].agg(
    lambda x: x.value_counts().index[0]
).to_dict()
print(f"Team mapping built: {len(team_mapping)} teams")

# For multi-candidate rows, keep only candidates whose team matches the injury team
multi_idx = n_cands[n_cands > 1].index
multi_rows = candidates[candidates['index'].isin(multi_idx)].copy()
multi_rows['Team_mapped'] = multi_rows['Team'].map(team_mapping)
multi_rows = multi_rows[multi_rows['Team_mapped'] == multi_rows['team']]

team_match_counts = multi_rows.groupby('index').size()
unique_team_idx = team_match_counts[team_match_counts == 1].index
team_lookup = (
    multi_rows[multi_rows['index'].isin(unique_team_idx)]
    .set_index('index')['playerId_mp']
)
injuries_clean['playerId'].update(team_lookup)

matched_final = injuries_clean['playerId'].notna().sum()
unmatched_final = injuries_clean['playerId'].isna().sum()
print(f"\nFinal: Matched {matched_final}, Unmatched {unmatched_final} ({matched_final / len(injuries_clean) * 100:.1f}%)")
print(f"\nRemaining unmatched rows:")
print(injuries_clean[injuries_clean['playerId'].isna()][['Player', 'season', 'Team', 'Position']].to_string())

Team mapping built: 35 teams

Final: Matched 15013, Unmatched 36 (99.8%)

Remaining unmatched rows:
                      Player  season          Team     Position
6834            Larsen, Brad    2008       Anaheim            F
6960         Warrener, Rhett    2008       Calgary            D
7468         Hatcher, Derian    2008  Philadelphia  D "Retired"
7469            Rathje, Mike    2008  Philadelphia  D "Retired"
8471       Picard, Alexandre    2009        Ottawa            D
8498            Rathje, Mike    2009  Philadelphia  D "Retired"
9004            Holøs, Jonas    2010      Colorado            D
10859         Bärtschi, Sven    2012       Calgary            F
12541         Bødker, Mikkel    2014       Arizona            F
12630         Bärtschi, Sven    2014       Calgary            F
13270          Müller, Mirco    2014      San Jose            D
13271          Müller, Mirco    2014      San Jose            D
14340         Bärtschi, Sven    2015     Vancouver            F
1434

/tmp/ipykernel_10301/3233844079.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  injuries_clean['playerId'].update(single_lookup)


In [ ]:
# Pass 4: manual map for transliteration/nickname cases, then drop intermediate columns
manual_map = {
    'Bärtschi, Sven': 8476466,    # Sven Baertschi (ä -> ae)
    'Bļugers, Teodors': 8476927,  # Teddy Blueger (nickname + spelling)
    'Bødker, Mikkel': 8474571,    # Mikkel Boedker (ø -> oe)
    'Holøs, Jonas': 8474730,      # Jonas Holos (ø -> o)
    'Müller, Mirco': 8477509,     # Mirco Mueller (ü -> ue)
    'Røndbjerg, Jonas': 8480007,  # Jonas Rondbjerg (ø -> o)
    'Steen, Alex': 8470257,       # Alexander Steen (nickname)
}

for player_str, pid in manual_map.items():
    mask = (injuries_clean['Player'] == player_str) & (injuries_clean['playerId'].isna())
    injuries_clean.loc[mask, 'playerId'] = pid

# Pettersson (F): two Elias Petterssons exist, pick the forward (the Vancouver star)
ep_forward_pid = df_combined_all[
    (df_combined_all['name'] == 'Elias Pettersson') &
    (df_combined_all['position'].isin(['C', 'L', 'R']))
]['playerId'].iloc[0]
mask = (injuries_clean['Player'] == 'Pettersson (F), Elias') & (injuries_clean['playerId'].isna())
injuries_clean.loc[mask, 'playerId'] = ep_forward_pid

# Picard 2009 Ottawa: two Alexandre Picards exist, pick the one who played for OTT
picard_ott_pid = df_combined_all[
    (df_combined_all['name'] == 'Alexandre Picard') &
    (df_combined_all['team'] == 'OTT')
]['playerId'].iloc[0]
mask = (
    (injuries_clean['Player'] == 'Picard, Alexandre') &
    (injuries_clean['season'] == 2009) &
    (injuries_clean['Team'] == 'Ottawa') &
    (injuries_clean['playerId'].isna())
)
injuries_clean.loc[mask, 'playerId'] = picard_ott_pid

matched = injuries_clean['playerId'].notna().sum()
unmatched = injuries_clean['playerId'].isna().sum()
print(f"After manual mapping: {matched} matched, {unmatched} unmatched ({matched / len(injuries_clean) * 100:.2f}%)")

# Remaining unmatched are retirees absent from df_combined_all, so we drop them
if unmatched > 0:
    print("\nUnmatched rows (retirees not in df_combined_all, dropping):")
    print(injuries_clean[injuries_clean['playerId'].isna()][['Player', 'season', 'Team', 'Position']])

before = len(injuries_clean)
injuries_clean = injuries_clean.dropna(subset=['playerId']).copy()
injuries_clean['playerId'] = injuries_clean['playerId'].astype(int)
print(f"\nDropped {before - len(injuries_clean)} orphan rows")

to_drop = [c for c in ['name_flipped', 'name_norm', 'last_norm', 'first_initial_inj'] if c in injuries_clean.columns]
injuries_clean = injuries_clean.drop(columns=to_drop)
print(f"Final injuries_clean shape: {injuries_clean.shape}")
print(f"Final columns: {injuries_clean.columns.tolist()}")

After manual mapping: 15044 matched, 5 unmatched (99.97%)

Unmatched rows (retirees not in df_combined_all, dropping):
               Player  season          Team     Position
6834     Larsen, Brad    2008       Anaheim            F
6960  Warrener, Rhett    2008       Calgary            D
7468  Hatcher, Derian    2008  Philadelphia  D "Retired"
7469     Rathje, Mike    2008  Philadelphia  D "Retired"
8498     Rathje, Mike    2009  Philadelphia  D "Retired"

Dropped 5 orphan rows
Final injuries_clean shape: (15044, 7)
Final columns: ['Team', 'Position', 'Player', 'Injury Type', 'Games Missed', 'season', 'playerId']


## Step 4. Aggregate injuries by player-season

We collapse the cleaned injury events into one summary row per player-season with an injury count, total games missed, and a breakdown of injury types.

In [ ]:
injuries_per_season = injuries_clean.groupby(['playerId', 'season']).agg(
    injury_count=('Injury Type', 'size'),
    games_missed_total=('Games Missed', 'sum'),
    injury_types=('Injury Type', lambda x: dict(x.value_counts())),
).reset_index()

print("Bedard's rows:")
injuries_per_season[injuries_per_season['playerId'] == 8484144]

Bedard's rows:


,playerId,season,injury_count,games_missed_total,injury_types
8826,8484144,2023,1,14,{'Jaw': 1}
8827,8484144,2025,2,13,"{'Illness': 1, 'Shoulder': 1}"


## Step 5. Build the final modeling table

We select the locked MoneyPuck schema, join the bios and injury aggregates, then engineer the rest. We add age at the start of the season, the games available that season, the share of games missed, draft fields with an undrafted flag, and the next-season games-missed target. We keep the latest season even though its target is unknown, so it stays available for clustering and EDA.

In [ ]:
# Select MoneyPuck columns from the locked schema
explicit_cols = [
    'playerId', 'season', 'name', 'team', 'position', 'games_played', 'icetime', 'shifts', 'gameScore', 'penalties',
    'onIce_xGoalsPercentage', 'onIce_corsiPercentage', 'onIce_fenwickPercentage', 'iceTimeRank', 'faceoffsWon', 'faceoffsLost',
    'timeOnBench', 'penalityMinutes', 'penalityMinutesDrawn', 'penaltiesDrawn', 'shotsBlockedByPlayer'
]
i_f_cols = sorted(c for c in df_combined_all.columns if c.startswith('I_F_'))
mp_cols = [c for c in explicit_cols + i_f_cols if c in df_combined_all.columns]
final = df_combined_all[mp_cols].copy()
final = final.drop(columns=[c for c in final.columns if '_x' in c or 'x' in c])

# Join bios
bio_cols = ['playerId', 'birthDate', 'heightInInches', 'weightInPounds', 'shootsCatches', 'draftYear', 'draftRound', 'draftOverallPick']
final = final.merge(player_bios[bio_cols], on='playerId', how='left')

# Join injury aggregates (no injuries that season means zero)
final = final.merge(injuries_per_season, on=['playerId', 'season'], how='left')
final['injury_count'] = final['injury_count'].fillna(0).astype(int)
final['games_missed_total'] = final['games_missed_total'].fillna(0).astype(int)

# Games available per season (default 82; shortened seasons noted)
GAMES_IN_SEASON = {
    2012: 48,  # 2012-13 lockout
    2019: 70,  # 2019-20 COVID, teams played 68-71 (~70 used as approx)
    2020: 56,  # 2020-21 COVID
}
final['games_in_season'] = final['season'].map(GAMES_IN_SEASON).fillna(82).astype(int)
final['percentage_games_missed'] = (final['games_missed_total'] / final['games_in_season']).round(3)

# Age at the start of the season (Oct 1 of the start year)
final['birthDate'] = pd.to_datetime(final['birthDate'])
season_start = pd.to_datetime(final['season'].astype(str) + '-10-01')
final['age_at_season'] = ((season_start - final['birthDate']).dt.days / 365.25).round(1)
final = final.drop(columns=['birthDate'])

# Target: next season's games missed. Sort first so the shift stays within each player.
final = final.sort_values(['playerId', 'season'])
final['percentage_games_missed_next_season'] = final.groupby('playerId')['percentage_games_missed'].shift(-1)

# Draft fields: fill undrafted with 0 and flag them
final['draftYear'] = final['draftYear'].fillna(0).astype(int)
final['draftRound'] = final['draftRound'].fillna(0).astype(int)
final['draftOverallPick'] = final['draftOverallPick'].fillna(0).astype(int)
final['undrafted'] = (final['draftYear'] == 0).astype(int)

# Onehotencode columns 'shootsCatches' and 'position'
final = pd.get_dummies(final, columns=['shootsCatches', 'position'], dtype=int)

# Feature Engineer columns "three_season_running_injuries" and "career_games_missed_so_far"
final['three_season_running_injuries'] = final.groupby('playerId')['injury_count'].transform(lambda x: x.rolling(3, min_periods=1).sum())
final['career_games_missed_so_far'] = final.groupby('playerId')['games_missed_total'].transform(lambda x: x.cumsum())

#final = final.sort_values(['playerId', 'season']).reset_index(drop=True)

print(f"Final shape: {final.shape}")
print(f"Columns ({len(final.columns)}): {final.columns.tolist()}")

Final shape: (16271, 80)
Columns (80): ['playerId', 'season', 'name', 'team', 'games_played', 'icetime', 'shifts', 'gameScore', 'penalties', 'onIce_corsiPercentage', 'onIce_fenwickPercentage', 'iceTimeRank', 'faceoffsWon', 'faceoffsLost', 'timeOnBench', 'penalityMinutes', 'penalityMinutesDrawn', 'penaltiesDrawn', 'shotsBlockedByPlayer', 'I_F_blockedShotAttempts', 'I_F_dZoneGiveaways', 'I_F_dZoneShiftEnds', 'I_F_dZoneShiftStarts', 'I_F_faceOffsWon', 'I_F_flyShiftEnds', 'I_F_flyShiftStarts', 'I_F_freeze', 'I_F_giveaways', 'I_F_goals', 'I_F_highDangerGoals', 'I_F_highDangerShots', 'I_F_hits', 'I_F_lowDangerGoals', 'I_F_lowDangerShots', 'I_F_mediumDangerGoals', 'I_F_mediumDangerShots', 'I_F_missedShots', 'I_F_neutralZoneShiftEnds', 'I_F_neutralZoneShiftStarts', 'I_F_oZoneShiftEnds', 'I_F_oZoneShiftStarts', 'I_F_penalityMinutes', 'I_F_playContinuedInZone', 'I_F_playContinuedOutsideZone', 'I_F_playStopped', 'I_F_points', 'I_F_primaryAssists', 'I_F_reboundGoals', 'I_F_rebounds', 'I_F_savedSho

In [ ]:
final.head(30)

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far
135,8445550,2008,Rob Blake,S.J,73,93140.0,1961.0,52.25,41.0,0.54,0.54,232.0,0.0,0.0,173750.0,88.0,23.0,10.0,114.0,117.0,22.0,258.0,266.0,0.0,1212.0,1091.0,52.0,37.0,10.0,4.0,9.0,66.0,5.0,217.0,1.0,22.0,50.0,301.0,357.0,190.0,247.0,88.0,81.0,82.0,3.0,45.0,10.0,3.0,20.0,188.0,238.0,366.07,248.75,25.0,1961.0,365.0,198.0,23.0,248.0,76.0,220.0,1988,4,70,3,9,"{'Foot': 1, 'Jaw': 1, 'Lower body': 1}",82,0.110,38.8,0.146,0,0,1,0,1,0,0,3.0,9
1265,8445550,2009,Rob Blake,S.J,69,88447.0,1962.0,35.90,28.0,0.49,0.49,201.0,0.0,0.0,163717.0,58.0,20.0,10.0,113.0,98.0,33.0,247.0,318.0,0.0,1214.0,1114.0,49.0,51.0,7.0,4.0,8.0,68.0,3.0,216.0,0.0,13.0,61.0,276.0,332.0,225.0,198.0,58.0,72.0,87.0,6.0,30.0,6.0,3.0,16.0,169.0,230.0,335.08,237.75,17.0,1962.0,335.0,176.0,19.0,237.0,76.0,220.0,1988,4,70,2,12,"{'Arm': 1, 'Undisclosed': 1}",82,0.146,39.8,NaN,0,0,1,0,1,0,0,5.0,21
330,8445735,2008,Rod Brind'Amour,CAR,80,91077.0,1899.0,46.75,18.0,0.53,0.52,285.0,908.0,580.0,200586.0,36.0,34.0,17.0,35.0,45.0,12.0,231.0,297.0,908.0,1051.0,958.0,31.0,40.0,16.0,8.0,16.0,76.0,6.0,133.0,2.0,34.0,48.0,335.0,303.0,282.0,341.0,36.0,56.0,70.0,3.0,51.0,16.0,4.0,7.0,119.0,167.0,229.65,184.52,19.0,1899.0,228.0,135.0,40.0,183.0,73.0,205.0,1988,1,9,1,2,{'Lower body': 1},82,0.024,38.1,0.000,0,1,0,1,0,0,0,1.0,2
1495,8445735,2009,Rod Brind'Amour,CAR,79,59845.0,1310.0,21.75,18.0,0.51,0.51,657.0,490.0,348.0,228556.0,36.0,30.0,14.0,31.0,32.0,11.0,189.0,193.0,490.0,709.0,707.0,21.0,28.0,9.0,5.0,15.0,82.0,3.0,85.0,1.0,32.0,37.0,219.0,194.0,193.0,216.0,36.0,45.0,49.0,3.0,19.0,5.0,2.0,5.0,86.0,123.0,166.00,133.09,5.0,1310.0,164.0,95.0,27.0,132.0,73.0,205.0,1988,1,9,0,0,NaN,82,0.000,39.1,NaN,0,1,0,1,0,0,0,1.0,2
718,8445739,2008,Patrice Brisebois,MTL,62,59144.0,1247.0,16.95,8.0,0.54,0.53,323.0,0.0,0.0,168371.0,19.0,21.0,4.0,69.0,72.0,47.0,173.0,114.0,0.0,707.0,750.0,11.0,54.0,5.0,2.0,4.0,45.0,2.0,84.0,1.0,3.0,23.0,178.0,189.0,189.0,194.0,19.0,34.0,34.0,1.0,18.0,4.0,0.0,6.0,63.0,86.0,163.48,91.04,9.0,1247.0,163.0,68.0,14.0,91.0,74.0,197.0,1989,2,30,1,4,{'Lower body': 1},82,0.049,37.7,NaN,0,0,1,0,1,0,0,1.0,4
831,8446053,2008,Chris Chelios,DET,28,19608.0,450.0,5.55,8.0,0.53,0.50,159.0,0.0,0.0,82814.0,18.0,14.0,7.0,12.0,14.0,8.0,62.0,38.0,0.0,275.0,289.0,1.0,13.0,0.0,0.0,0.0,22.0,0.0,23.0,0.0,1.0,10.0,52.0,66.0,61.0,57.0,18.0,12.0,10.0,1.0,0.0,0.0,0.0,0.0,14.0,24.0,38.48,24.43,0.0,450.0,38.0,14.0,7.0,24.0,72.0,191.0,1981,2,40,1,28,{'Leg': 1},82,0.341,46.7,0.000,0,0,1,0,1,0,0,1.0,28
1093,8446053,2009,Chris Chelios,ATL,7,4693.0,106.0,1.63,0.0,0.52,0.48,43.0,0.0,0.0,21016.0,0.0,2.0,1.0,7.0,4.0,4.0,15.0,14.0,0.0,60.0,67.0,0.0,5.0,0.0,0.0,0.0,8.0,0.

### Step 5a. Validation

We spot-check a known player and review the final column set.

In [ ]:
crosby = final[final['name'] == 'Sidney Crosby']
crosby

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far
83,8471675,2008,Sidney Crosby,PIT,77,101373.0,1835.0,93.58,29.0,0.57,0.58,155.0,829.0,786.0,181120.0,66.0,92.0,43.0,42.0,69.0,15.0,274.0,185.0,829.0,952.0,832.0,40.0,80.0,33.0,9.0,24.0,80.0,9.0,196.0,15.0,99.0,81.0,377.0,384.0,232.0,434.0,66.0,98.0,116.0,12.0,103.0,51.0,1.0,20.0,205.0,286.0,392.03,321.44,19.0,1835.0,388.0,238.0,56.0,319.0,71.0,200.0,2005,1,1,2,5,"{'Groin': 1, 'Lower body': 1}",82,0.061,21.2,0.012,0,1,0,1,0,0,0,2.0,5
1763,8471675,2009,Sidney Crosby,PIT,80,105401.0,1876.0,116.70,32.0,0.62,0.62,129.0,993.0,781.0,186817.0,67.0,71.0,34.0,42.0,97.0,19.0,236.0,249.0,993.0,992.0,855.0,49.0,77.0,51.0,16.0,29.0,62.0,21.0,267.0,14.0,97.0,102.0,406.0,359.0,242.0,413.0,67.0,137.0,124.0,13.0,109.0,36.0,5.0,19.0,240.0,342.0,490.40,394.35,22.0,1876.0,490.0,291.0,43.0,393.0,71.0,200.0,2005,1,1,1,1,{'Groin': 1},82,0.012,22.2,0.500,0,1,0,1,0,0,0,3.0,6
2197,8471675,2010,Sidney Crosby,PIT,41,53917.0,930.0,67.69,14.0,0.62,0.61,50.0,546.0,435.0,95491.0,31.0,47.0,23.0,23.0,59.0,7.0,103.0,143.0,546.0,487.0,407.0,27.0,29.0,32.0,12.0,21.0,31.0,8.0,142.0,12.0,56.0,58.0,208.0,186.0,132.0,194.0,31.0,73.0,67.0,13.0,66.0,20.0,5.0,7.0,129.0,187.0,282.56,221.75,14.0,930.0,278.0,161.0,15.0,219.0,71.0,200.0,2005,1,1,1,41,{'Concussion': 1},82,0.500,23.2,0.732,0,1,0,1,0,0,0,4.0,47
2956,8471675,2011,Sidney Crosby,PIT,22,24383.0,453.0,37.69,7.0,0.66,0.66,73.0,227.0,226.0,55492.0,14.0,12.0,6.0,7.0,29.0,6.0,77.0,52.0,227.0,189.0,203.0,15.0,15.0,8.0,3.0,13.0,11.0,2.0,53.0,3.0,31.0,22.0,117.0,88.0,70.0,110.0,14.0,30.0,38.0,3.0,37.0,16.0,2.0,3.0,67.0,89.0,129.37,99.04,13.0,453.0,126.0,75.0,10.0,97.0,71.0,200.0,2005,1,1,2,60,{'Concussion': 2},82,0.732,24.2,0.250,0,1,0,1,0,0,0,4.0,107
3586,8471675,2012,Sidney Crosby,PIT,36,45585.0,820.0,61.77,8.0,0.62,0.62,61.0,453.0,381.0,84820.0,16.0,34.0,17.0,18.0,35.0,5.0,96.0,142.0,453.0,416.0,358.0,24.0,28.0,15.0,4.0,12.0,21.0,4.0,103.0,7.0,53.0,44.0,189.0,164.0,119.0,156.0,16.0,63.0,48.0,4.0,56.0,28.0,4.0,14.0,109.0,153.0,206.41,170.50,13.0,820.0,203.0,124.0,15.0,168.0,71.0,200.0,2005,1,1,1,12,{'Facial': 1},48,0.250,25.2,0.012,0,1,0,1,0,0,0,4.0,119
5242,8471675,2013,Sidney Crosby,PIT,80,105467.0,1917.0,102.87,22.0,0.60,0.61,104.0,991.0,896.0,185946.0,46.0,62.0,32.0,28.0,77.0,12.0,229.0,264.0,991.0,1062.0,903.0,41.0,56.0,36.0,12.0,28.0,47.0,9.0,223.0,15.0,93.0,85.0,395.0,371.0,231.0,379.0,46.0,110.0,123.0,13.0,104.0,34.0,5.0,21.0,223.0,308.0,424.05,346.58,34.0,1917.0,421.0,259.0,41.0,344.0,71.0,200.0,2005,1,1,1,1,{'Upper body': 1},82,0.012,26.2,0.061,0,1,0,1,0,0,0,4.0,120
6059,8471675,2014,Sidney Crosby,PIT,77,92210.0,1926.0,92.55,17.0,0.62,0.62,141.0,797.0,800.0,189584

In [ ]:
sorted(final.columns)

['I_F_blockedShotAttempts',
 'I_F_dZoneGiveaways',
 'I_F_dZoneShiftEnds',
 'I_F_dZoneShiftStarts',
 'I_F_faceOffsWon',
 'I_F_flyShiftEnds',
 'I_F_flyShiftStarts',
 'I_F_freeze',
 'I_F_giveaways',
 'I_F_goals',
 'I_F_highDangerGoals',
 'I_F_highDangerShots',
 'I_F_hits',
 'I_F_lowDangerGoals',
 'I_F_lowDangerShots',
 'I_F_mediumDangerGoals',
 'I_F_mediumDangerShots',
 'I_F_missedShots',
 'I_F_neutralZoneShiftEnds',
 'I_F_neutralZoneShiftStarts',
 'I_F_oZoneShiftEnds',
 'I_F_oZoneShiftStarts',
 'I_F_penalityMinutes',
 'I_F_playContinuedInZone',
 'I_F_playContinuedOutsideZone',
 'I_F_playStopped',
 'I_F_points',
 'I_F_primaryAssists',
 'I_F_reboundGoals',
 'I_F_rebounds',
 'I_F_savedShotsOnGoal',
 'I_F_savedUnblockedShotAttempts',
 'I_F_scoreAdjustedShotsAttempts',
 'I_F_scoreAdjustedUnblockedShotAttempts',
 'I_F_secondaryAssists',
 'I_F_shifts',
 'I_F_shotAttempts',
 'I_F_shotsOnGoal',
 'I_F_takeaways',
 'I_F_unblockedShotAttempts',
 'age_at_season',
 'career_games_missed_so_far',
 'draf

## Step 6. Add salary and cap hit from CapWages

After meeting with Erik we add contract data from CapWages. Each player page embeds contract JSON in a script tag, so we parse per-season cap hit, base and total salary, clause, and expiry status, and compute cap percentage against that season's ceiling. The scrape is cached to player_salaries.csv and is resumable.

In [ ]:
# NHL upper cap limit by season start year ($)
CAP_CEILING = {
    2005: 39_000_000, 2006: 44_000_000, 2007: 50_300_000, 2008: 56_700_000,
    2009: 56_800_000, 2010: 59_400_000, 2011: 64_300_000, 2012: 60_000_000,
    2013: 64_300_000, 2014: 69_000_000, 2015: 71_400_000, 2016: 73_000_000,
    2017: 75_000_000, 2018: 79_500_000, 2019: 81_500_000, 2020: 81_500_000,
    2021: 81_500_000, 2022: 82_500_000, 2023: 83_500_000, 2024: 88_000_000,
    2025: 95_500_000, 2026: 104_000_000, 2027: 113_500_000,
}

def _money(x):
    if x is None: return None
    x = str(x).replace('$', '').replace(',', '').strip()
    if x in ('', '-', '—', 'None'): return None
    try: return int(x)
    except:
        try: return float(x)
        except: return None

def _extract_array(s, key):
    # String-aware balanced-bracket extractor for the embedded "contracts" JSON array
    i = s.find(key)
    if i < 0: return None
    j = s.find('[', i); depth = 0; in_str = False; esc = False
    for k in range(j, len(s)):
        c = s[k]
        if in_str:
            if esc: esc = False
            elif c == '\\': esc = True
            elif c == '"': in_str = False
        elif c == '"': in_str = True
        elif c == '[': depth += 1
        elif c == ']':
            depth -= 1
            if depth == 0:
                try: return json.loads(s[j:k+1])
                except Exception: return None
    return None

def name_to_slug(name):
    # Reuse normalize_name, then hyphenate everything non-alphanumeric (keeps O'Reilly -> o-reilly)
    norm = normalize_name(name)
    if norm is None:
        return None
    return re.sub(r'[^a-z0-9]+', '-', norm).strip('-')

def fetch_capwages(slug):
    url = f"https://capwages.com/players/{slug}"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    scripts = re.findall(r'<script[^>]*>(.*?)</script>', r.text, re.S)
    cand = [x for x in scripts if 'capHit' in x]
    if not cand: return None
    dec = ihtml.unescape(max(cand, key=lambda x: x.count('capHit')))
    contracts = _extract_array(dec, '"contracts":') or []
    rows = []
    for c in contracts:
        for d in c.get('details', []):
            seas = str(d.get('season', ''))
            if not seas[:4].isdigit(): continue
            yr = int(seas[:4])
            cap_hit = _money(d.get('capHit'))
            ceiling = CAP_CEILING.get(yr)
            cap_pct = round(cap_hit / ceiling * 100, 1) if cap_hit and ceiling else None
            rows.append({
                'season': yr,
                'cap_hit': cap_hit,
                'base_salary': _money(d.get('baseSalary')),
                'total_salary': _money(d.get('totalSalary')),
                'cap_pct': cap_pct,
                'cap_clause': d.get('clause') or None,
                'expiry_status': c.get('expiryStatus') or None,
            })
    return rows

In [ ]:
# Sanity check on a known player
slug = name_to_slug('Sidney Crosby')
crosby_salary = pd.DataFrame(fetch_capwages(slug)).sort_values('season').reset_index(drop=True)
crosby_salary

,season,cap_hit,base_salary,total_salary,cap_pct,cap_clause,expiry_status
0,2005,850000,765000,850000,2.2,None,RFA
1,2006,850000,765000,850000,1.9,None,RFA
2,2007,850000,765000,850000,1.7,None,RFA
3,2008,8700000,9000000,9000000,15.3,None,UFA
4,2009,8700000,9000000,9000000,15.3,None,UFA
5,2010,8700000,9000000,9000000,14.6,None,UFA
6,2011,8700000,9000000,9000000,13.5,None,UFA
7,2012,8700000,7500000,7500000,14.5,NMC,UFA
8,2013,8700000,12000000,12000000,13.5,NMC,UFA
9,2014,8700000,12000000,12000000,12.6,NMC,UFA


In [ ]:
%%script false --no-raise-error
salary_cache = f"{path}/player_salaries.csv"
SAL_COLS = ['playerId', 'season', 'cap_hit', 'base_salary', 'total_salary', 'cap_pct', 'cap_clause', 'expiry_status']

# Manual slug overrides for nickname/formal-name cases (playerId -> CapWages slug)
SLUG_OVERRIDES = {
    8466285: 'brian-campbell', 8468542: 'paul-martin', 8468575: 'dominic-moore',
    8469500: 'mike-cammalleri', 8469707: 'pierre-alexandre-parenteau', 8470640: 'matthew-carle',
    8474018: 'chris-didomenico', 8474089: 'brad-malone', 8475164: 'jakob-silfverberg',
    8475321: 'phil-varone', 8475625: 'matt-irwin', 8475735: 'greg-mckegg', 8475766: 'austin-watson',
    8476326: 'josh-archibald', 8476396: 'steven-fogarty', 8476404: 'andy-andreoff',
    8476452: 'tyler-wotherspoon', 8476495: 'seth-griffith', 8476505: 'turner-elson',
    8476851: 'alex-galchenyuk', 8476915: 'jujhar-khaira', 8476922: 'boo-nieves',
    8476953: 'trevor-carrick', 8476982: 'danny-o-regan', 8477046: 'joakim-ryan',
    8477085: 'ashton-sautner', 8477215: 'danny-dekeyser', 8477401: 'john-hayden',
    8477464: 'nic-petan', 8477471: 'dillon-heatherington', 8477482: 'zachary-sanford',
    8477520: 'jean-sebastien-dea', 8477972: 'josh-jacobs', 8477981: 'roland-mckeown',
    8478029: 'chase-de-leo', 8478427: 'sebastian-aho', 8478442: 'cooper-marody',
    8478447: 'nick-merkley', 8478996: 'adam-brooks', 8479252: 'dan-renouf',
    8479304: 'kyle-criscuolo', 8479322: 'wade-allison', 8479338: 'markus-niemelainen',
    8479366: 'logan-brown', 8479381: 'nicholas-caamano', 8479526: 'otto-koivula',
    8479547: 'matthew-phillips', 8479573: 'mikhail-maltsev', 8480008: 'adam-ruzicka',
    8480031: 'maxime-comtois', 8480072: 'lias-andersson', 8480083: 'c-j-smith',
    8480160: 'radim-simek', 8480216: 'jacob-peterson', 8480226: 'marian-studenic',
    8480384: 'alexander-true', 8480443: 'kristian-reichel', 8480776: 'sheldon-rempal',
    8480814: 'nicolas-beaudin', 8480840: 'oskar-back', 8480913: 'jordan-gross',
    8481004: 'josiah-slavin', 8481594: 'nathan-legare', 8481690: 'tyler-angle',
    8481720: 'nick-abruzzese', 8482947: 'zakhar-bardakov', 8482953: 'taylor-makar',
    8483476: 'jonathan-lekkerimaki', 8483841: 'xavier-parent', 8484305: 'zach-metsa',
    8484794: 'michael-brandsegg-nygard', 8485493: 'david-tomasek', 8485512: 'tim-washe',
}
players = final[['playerId', 'name']].drop_duplicates().reset_index(drop=True)
players['slug'] = players['playerId'].map(SLUG_OVERRIDES).fillna(players['name'].apply(name_to_slug))

if os.path.exists(salary_cache) and os.path.getsize(salary_cache) > 1:
    player_salaries = pd.read_csv(salary_cache)
    done_ids = set(player_salaries['playerId'].unique())
else:
    player_salaries = pd.DataFrame(columns=SAL_COLS)
    done_ids = set()

todo = players[~players['playerId'].isin(done_ids)]
print(f"{len(done_ids)} players cached, {len(todo)} to fetch")

new_rows, misses = [], []
bar = tqdm(list(todo.itertuples(index=False)), total=len(todo))
for n, row in enumerate(bar, start=1):
    data = fetch_capwages(row.slug)
    if not data:
        misses.append((row.playerId, row.name, row.slug))
    else:
        for d in data:
            d['playerId'] = row.playerId
            new_rows.append(d)
    bar.set_postfix(misses=len(misses))
    time.sleep(0.5)
    if n % 100 == 0:
        pd.concat([player_salaries, pd.DataFrame(new_rows)], ignore_index=True).to_csv(salary_cache, index=False)

player_salaries = pd.concat([player_salaries, pd.DataFrame(new_rows)], ignore_index=True)
player_salaries = player_salaries[SAL_COLS].drop_duplicates(subset=['playerId', 'season'])
player_salaries.to_csv(salary_cache, index=False)

# Merge salary onto final (drop first so the cell is idempotent; we don't fillna since 0 != $0)
add_cols = ['cap_hit', 'total_salary', 'cap_pct']
final = final.drop(columns=[c for c in add_cols if c in final.columns])
final = final.merge(player_salaries[['playerId', 'season'] + add_cols], on=['playerId', 'season'], how='left')

have = final['cap_hit'].notna().sum()
print(f"Salary rows: {len(player_salaries)} | cap_hit coverage: {have}/{len(final)} ({have / len(final) * 100:.1f}%) | misses: {len(misses)}")

In [ ]:
# cap_hit fill rate and usable row count by season
final = final.drop(columns=[c for c in final.columns if c.startswith(('cap_hit', 'total_salary', 'cap_pct'))])
final = final.merge(pd.read_csv(f"{path}/player_salaries.csv")[['playerId', 'season', 'cap_hit', 'total_salary', 'cap_pct']], on=['playerId', 'season'], how='left')

g = final.groupby('season')['cap_hit']
cap_hit_cov = pd.DataFrame({
    'pct_cov': (g.apply(lambda s: s.notna().mean()) * 100).round(1),
    'usable_rows': g.apply(lambda s: s.notna().sum()),
    'total_rows': g.size(),
})

print(cap_hit_cov)

        pct_cov  usable_rows  total_rows
season                                  
2008       59.5          527         885
2009       66.9          588         879
2010       73.0          650         891
2011       78.3          700         894
2012       84.7          711         839
2013       87.1          772         886
2014       91.3          805         882
2015       92.2          828         898
2016       93.5          829         887
2017       94.0          837         890
2018       93.4          846         906
2019       94.0          830         883
2020       94.2          860         913
2021       94.3          946        1003
2022       95.3          906         951
2023       98.7          912         924
2024       99.5          915         920
2025       99.0          931         940


In [ ]:
# 2015+ player-seasons still missing salary
gaps = final[(final['season'] >= 2015) & (final['cap_hit'].isna())]
gaps[['playerId', 'name', 'season', 'team']].drop_duplicates('playerId').sort_values('name')
gaps

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far,cap_hit,total_salary,cap_pct
294,8460542,2015,Patrik Elias,N.J,16,14819.0,340.0,7.08,5.0,0.54,0.55,97.0,9.0,19.0,43996.0,10.0,8.0,4.0,1.0,9.0,0.0,32.0,39.0,9.0,206.0,195.0,9.0,9.0,2.0,0.0,1.0,10.0,1.0,18.0,1.0,7.0,5.0,58.0,44.0,44.0,62.0,10.0,6.0,9.0,0.0,8.0,3.0,0.0,0.0,19.0,24.0,34.48,25.52,3.0,340.0,35.0,21.0,5.0,26.0,73.0,190.0,1994,2,51,2,66,{'Knee': 2},82,0.805,39.5,NaN,0,1,0,1,0,0,0,10.0,125,NaN,NaN,NaN
347,8460720,2015,Eric Boulton,NYI,6,3415.0,88.0,-1.23,1.0,0.34,0.29,65.0,0.0,0.0,18620.0,2.0,2.0,1.0,0.0,2.0,1.0,7.0,6.0,0.0,64.0,61.0,1.0,2.0,0.0,0.0,0.0,15.0,0.0,3.0,0.0,0.0,0.0,13.0,13.0,4.0,8.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,3.0,5.41,3.16,0.0,88.0,5.0,3.0,1.0,3.0,72.0,227.0,1994,9,234,2,46,"{'Lower body': 1, 'Upper body': 1}",82,0.561,39.1,NaN,0,1,0,0,0,1,0,5.0,116,NaN,NaN,NaN
469,8464977,2015,Dainius Zubrus,S.J,50,34769.0,873.0,6.95,10.0,0.46,0.47,513.0,15.0,12.0,146963.0,20.0,10.0,5.0,15.0,17.0,4.0,130.0,74.0,15.0,514.0,610.0,9.0,15.0,3.0,0.0,5.0,68.0,1.0,31.0,2.0,13.0,16.0,103.0,129.0,126.0,60.0,20.0,14.0,19.0,2.0,7.0,1.0,0.0,2.0,30.0,46.0,65.95,48.76,3.0,873.0,66.0,33.0,15.0,49.0,77.0,225.0,1996,1,15,2,7,"{'Undisclosed': 1, 'Upper body': 1}",82,0.085,37.3,NaN,0,1,0,1,0,0,0,3.0,75,NaN,NaN,NaN
587,8465185,2015,Willie Mitchell,FLA,46,54994.0,1093.0,0.27,9.0,0.36,0.37,154.0,0.0,0.0,113053.0,18.0,12.0,6.0,70.0,27.0,14.0,154.0,232.0,0.0,611.0,605.0,10.0,18.0,1.0,0.0,0.0,68.0,0.0,46.0,1.0,4.0,17.0,188.0,168.0,140.0,88.0,18.0,16.0,21.0,0.0,7.0,1.0,0.0,2.0,32.0,49.0,77.07,49.66,5.0,1093.0,77.0,33.0,6.0,50.0,75.0,210.0,1996,8,199,1,36,{'Upper body': 1},82,0.439,38.4,NaN,0,1,0,0,1,0,0,3.0,122,NaN,NaN,NaN
628,8465978,2015,Shawn Thornton,FLA,50,26067.0,593.0,5.19,21.0,0.48,0.49,563.0,6.0,7.0,156725.0,60.0,46.0,14.0,2.0,11.0,2.0,85.0,32.0,6.0,373.0,389.0,9.0,4.0,1.0,0.0,1.0,73.0,0.0,63.0,1.0,13.0,19.0,64.0,82.0,71.0,90.0,60.0,21.0,38.0,3.0,5.0,4.0,0.0,5.0,57.0,76.0,88.86,77.63,0.0,593.0,88.0,58.0,9.0,77.0,74.0,217.0,1997,7,190,0,0,NaN,82,0.000,38.2,0.0,0,0,1,0,0,1,0,3.0,36,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16243,8484976,2024,Jere Innala,COL,17,7293.0,166.0,0.04,1.0,0.46,0.50,187.0,1.0,2.0,54535.0,2.0,6.0,3.0,6.0,4.0,0.0,21.0,18.0,1.0,109.0,108.0,2.0,2.0,0.0,0.0,2.0,20.0,0.0,8.0,0.0,7.0,9.0,6.0,27.0,30.0,13.0,2.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,8.0,17.0,20.82,16.91,0.0,16

In [ ]:
# Final validation check
blake = final[final['name'] == 'Rob Blake']
blake

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far,cap_hit,total_salary,cap_pct
0,8445550,2008,Rob Blake,S.J,73,93140.0,1961.0,52.25,41.0,0.54,0.54,232.0,0.0,0.0,173750.0,88.0,23.0,10.0,114.0,117.0,22.0,258.0,266.0,0.0,1212.0,1091.0,52.0,37.0,10.0,4.0,9.0,66.0,5.0,217.0,1.0,22.0,50.0,301.0,357.0,190.0,247.0,88.0,81.0,82.0,3.0,45.0,10.0,3.0,20.0,188.0,238.0,366.07,248.75,25.0,1961.0,365.0,198.0,23.0,248.0,76.0,220.0,1988,4,70,3,9,"{'Foot': 1, 'Jaw': 1, 'Lower body': 1}",82,0.110,38.8,0.146,0,0,1,0,1,0,0,3.0,9,NaN,NaN,NaN
1,8445550,2009,Rob Blake,S.J,69,88447.0,1962.0,35.90,28.0,0.49,0.49,201.0,0.0,0.0,163717.0,58.0,20.0,10.0,113.0,98.0,33.0,247.0,318.0,0.0,1214.0,1114.0,49.0,51.0,7.0,4.0,8.0,68.0,3.0,216.0,0.0,13.0,61.0,276.0,332.0,225.0,198.0,58.0,72.0,87.0,6.0,30.0,6.0,3.0,16.0,169.0,230.0,335.08,237.75,17.0,1962.0,335.0,176.0,19.0,237.0,76.0,220.0,1988,4,70,2,12,"{'Arm': 1, 'Undisclosed': 1}",82,0.146,39.8,NaN,0,0,1,0,1,0,0,5.0,21,NaN,NaN,NaN


In [ ]:
washe = final[final['name'] == 'Tim Washe']
washe

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far,cap_hit,total_salary,cap_pct
16265,8485512,2024,Tim Washe,ANA,2,928.0,25.0,0.90,0.0,0.70,0.72,24.0,4.0,4.0,6625.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,4.0,4.0,12.0,21.0,1.0,0.0,0.0,0.0,1.0,4.0,0.0,4.0,0.0,1.0,4.0,2.0,0.0,7.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0,6.0,7.11,6.05,0.0,25.0,7.0,2.0,0.0,6.0,75.0,212.0,0,0,0,0,0,NaN,82,0.0,23.1,0.0,1,1,0,1,0,0,0,0.0,0,NaN,NaN,NaN
16266,8485512,2025,Tim Washe,ANA,39,28202.0,705.0,5.95,7.0,0.43,0.44,356.0,225.0,163.0,115211.0,14.0,12.0,6.0,25.0,15.0,3.0,96.0,148.0,225.0,443.0,409.0,4.0,11.0,2.0,1.0,4.0,91.0,0.0,38.0,1.0,19.0,28.0,90.0,93.0,76.0,55.0,14.0,25.0,21.0,1.0,5.0,2.0,2.0,8.0,31.0,59.0,74.72,59.92,1.0,705.0,76.0,33.0,12.0,61.0,75.0,212.0,0,0,0,0,0,NaN,82,0.0,24.1,NaN,1,1,0,1,0,0,0,0.0,0,812500.0,775000.0,0.9


In [ ]:
%%script false --no-raise-error
# Taylor's Ask (06/09/26)
before = len(final)
final = final[final['cap_hit'].notna()].reset_index(drop=True)
print(f"Dropped {before - len(final)} rows with no cap_hit")

In [ ]:
# Rows per season with no NaN in any column
complete_by_season = final.dropna().groupby('season').size()
print(complete_by_season)

season
2008    283
2009    312
2010    332
2011    381
2012    277
2013    398
2014    395
2015    414
2016    417
2017    406
2018    414
2019    390
2020    435
2021    591
2022    438
2023    426
2024    407
dtype: int64


In [ ]:

complete = final.dropna()
complete = complete[complete['season'] >= 2012]
complete.head()

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far,cap_hit,total_salary,cap_pct
13,8448208,2012,Jaromir Jagr,BOS,45,49491.0,931.0,37.06,11.0,0.59,0.57,158.0,0.0,1.0,113827.0,22.0,22.0,11.0,10.0,50.0,3.0,138.0,94.0,0.0,497.0,476.0,21.0,28.0,16.0,6.0,15.0,13.0,3.0,102.0,7.0,40.0,42.0,165.0,164.0,131.0,197.0,22.0,61.0,48.0,3.0,35.0,10.0,2.0,8.0,99.0,141.0,208.02,157.75,9.0,931.0,207.0,115.0,19.0,157.0,75.0,230.0,1990,1,5,2,3,"{'Flu': 1, 'Groin': 1}",48,0.062,40.6,0.000,0,1,0,0,0,0,1,5.0,8,4500000.0,4500000.0,7.5
15,8448208,2014,Jaromir Jagr,FLA,77,81174.0,1671.0,51.03,24.0,0.57,0.56,324.0,1.0,0.0,200512.0,48.0,36.0,18.0,14.0,74.0,14.0,239.0,172.0,1.0,898.0,894.0,38.0,46.0,17.0,5.0,17.0,30.0,6.0,143.0,6.0,66.0,57.0,281.0,305.0,253.0,300.0,48.0,84.0,71.0,1.0,47.0,21.0,4.0,15.0,152.0,209.0,301.39,226.34,9.0,1671.0,300.0,169.0,38.0,226.0,75.0,230.0,1990,1,5,3,5,"{'Illness': 2, 'Flu': 1}",82,0.061,42.6,0.037,0,1,0,0,0,0,1,5.0,13,3500000.0,3500000.0,5.1
16,8448208,2015,Jaromir Jagr,FLA,79,80936.0,1616.0,59.16,23.0,0.57,0.58,381.0,0.0,1.0,207214.0,48.0,24.0,12.0,16.0,44.0,11.0,238.0,132.0,0.0,866.0,950.0,27.0,44.0,27.0,4.0,15.0,31.0,9.0,131.0,14.0,64.0,67.0,266.0,291.0,246.0,243.0,48.0,70.0,72.0,6.0,66.0,21.0,2.0,8.0,116.0,183.0,256.28,211.45,18.0,1616.0,254.0,143.0,33.0,210.0,75.0,230.0,1990,1,5,2,3,"{'Flu': 1, 'Lower body': 1}",82,0.037,43.6,0.000,0,1,0,0,0,0,1,5.0,16,3500000.0,3500000.0,4.9
63,8457981,2012,Teemu Selanne,ANA,46,43311.0,871.0,20.57,4.0,0.53,0.54,266.0,85.0,83.0,125667.0,8.0,12.0,6.0,10.0,36.0,7.0,126.0,63.0,85.0,468.0,497.0,16.0,31.0,12.0,4.0,7.0,8.0,5.0,89.0,3.0,35.0,35.0,161.0,145.0,116.0,166.0,8.0,51.0,49.0,0.0,24.0,10.0,0.0,3.0,84.0,119.0,169.30,132.02,2.0,871.0,167.0,96.0,23.0,131.0,72.0,209.0,1988,1,10,1,1,{'Lower body': 1},48,0.021,42.2,0.049,0,0,1,0,0,0,1,4.0,53,4500000.0,4500000.0,7.5
124,8458951,2012,Sergei Gonchar,OTT,45,64737.0,1297.0,30.12,13.0,0.56,0.54,93.0,0.0,0.0,100301.0,26.0,4.0,2.0,65.0,71.0,20.0,177.0,159.0,0.0,732.0,694.0,23.0,28.0,3.0,1.0,2.0,25.0,2.0,114.0,0.0,4.0,35.0,218.0,223.0,170.0,221.0,26.0,43.0,42.0,3.0,27.0,9.0,1.0,6.0,82.0,117.0,190.87,119.77,15.0,1297.0,191.0,85.0,10.0,120.0,74.0,210.0,1992,1,14,2,3,"{'Flu': 1, 'Lower body': 1}",48,0.062,38.5,0.073,0,1,0,0,1,0,0,8.0,102,5500000.0,5500000.0,9.2


In [ ]:
import ast
from collections import Counter

# Adding Injury Mapping
mapping = pd.read_csv("/content/drive/MyDrive/SIADS 696: Milestone II/injury_type_mapping.csv")
# mapping.head()

region = dict(zip(mapping['Injury Type'].str.strip().str.lower(), mapping['Injury Region'].str.strip().str.lower()))
# region

def region_counts(d):
    d = ast.literal_eval(d) if isinstance(d, str) else d
    d = d if isinstance(d, dict) else {}
    c = Counter()
    for inj, n in d.items():
        r = region.get(str(inj).strip().lower())
        if r: c[f"injury_{r}"] += n
    return c


region_df = complete['injury_types'].apply(region_counts).apply(pd.Series).fillna(0).astype(int)

complete = pd.concat([complete.drop(columns=[c for c in region_df.columns if c in complete.columns]), region_df], axis=1)
complete.head()

,playerId,season,name,team,games_played,icetime,shifts,gameScore,penalties,onIce_corsiPercentage,onIce_fenwickPercentage,iceTimeRank,faceoffsWon,faceoffsLost,timeOnBench,penalityMinutes,penalityMinutesDrawn,penaltiesDrawn,shotsBlockedByPlayer,I_F_blockedShotAttempts,I_F_dZoneGiveaways,I_F_dZoneShiftEnds,I_F_dZoneShiftStarts,I_F_faceOffsWon,I_F_flyShiftEnds,I_F_flyShiftStarts,I_F_freeze,I_F_giveaways,I_F_goals,I_F_highDangerGoals,I_F_highDangerShots,I_F_hits,I_F_lowDangerGoals,I_F_lowDangerShots,I_F_mediumDangerGoals,I_F_mediumDangerShots,I_F_missedShots,I_F_neutralZoneShiftEnds,I_F_neutralZoneShiftStarts,I_F_oZoneShiftEnds,I_F_oZoneShiftStarts,I_F_penalityMinutes,I_F_playContinuedInZone,I_F_playContinuedOutsideZone,I_F_playStopped,I_F_points,I_F_primaryAssists,I_F_reboundGoals,I_F_rebounds,I_F_savedShotsOnGoal,I_F_savedUnblockedShotAttempts,I_F_scoreAdjustedShotsAttempts,I_F_scoreAdjustedUnblockedShotAttempts,I_F_secondaryAssists,I_F_shifts,I_F_shotAttempts,I_F_shotsOnGoal,I_F_takeaways,I_F_unblockedShotAttempts,heightInInches,weightInPounds,draftYear,draftRound,draftOverallPick,injury_count,games_missed_total,injury_types,games_in_season,percentage_games_missed,age_at_season,percentage_games_missed_next_season,undrafted,shootsCatches_L,shootsCatches_R,position_C,position_D,position_L,position_R,three_season_running_injuries,career_games_missed_so_far,cap_hit,total_salary,cap_pct,injury_illness,injury_lower,injury_head,injury_upper,injury_undisclosed,injury_internal
13,8448208,2012,Jaromir Jagr,BOS,45,49491.0,931.0,37.06,11.0,0.59,0.57,158.0,0.0,1.0,113827.0,22.0,22.0,11.0,10.0,50.0,3.0,138.0,94.0,0.0,497.0,476.0,21.0,28.0,16.0,6.0,15.0,13.0,3.0,102.0,7.0,40.0,42.0,165.0,164.0,131.0,197.0,22.0,61.0,48.0,3.0,35.0,10.0,2.0,8.0,99.0,141.0,208.02,157.75,9.0,931.0,207.0,115.0,19.0,157.0,75.0,230.0,1990,1,5,2,3,"{'Flu': 1, 'Groin': 1}",48,0.062,40.6,0.000,0,1,0,0,0,0,1,5.0,8,4500000.0,4500000.0,7.5,1,1,0,0,0,0
15,8448208,2014,Jaromir Jagr,FLA,77,81174.0,1671.0,51.03,24.0,0.57,0.56,324.0,1.0,0.0,200512.0,48.0,36.0,18.0,14.0,74.0,14.0,239.0,172.0,1.0,898.0,894.0,38.0,46.0,17.0,5.0,17.0,30.0,6.0,143.0,6.0,66.0,57.0,281.0,305.0,253.0,300.0,48.0,84.0,71.0,1.0,47.0,21.0,4.0,15.0,152.0,209.0,301.39,226.34,9.0,1671.0,300.0,169.0,38.0,226.0,75.0,230.0,1990,1,5,3,5,"{'Illness': 2, 'Flu': 1}",82,0.061,42.6,0.037,0,1,0,0,0,0,1,5.0,13,3500000.0,3500000.0,5.1,3,0,0,0,0,0
16,8448208,2015,Jaromir Jagr,FLA,79,80936.0,1616.0,59.16,23.0,0.57,0.58,381.0,0.0,1.0,207214.0,48.0,24.0,12.0,16.0,44.0,11.0,238.0,132.0,0.0,866.0,950.0,27.0,44.0,27.0,4.0,15.0,31.0,9.0,131.0,14.0,64.0,67.0,266.0,291.0,246.0,243.0,48.0,70.0,72.0,6.0,66.0,21.0,2.0,8.0,116.0,183.0,256.28,211.45,18.0,1616.0,254.0,143.0,33.0,210.0,75.0,230.0,1990,1,5,2,3,"{'Flu': 1, 'Lower body': 1}",82,0.037,43.6,0.000,0,1,0,0,0,0,1,5.0,16,3500000.0,3500000.0,4.9,1,1,0,0,0,0
63,8457981,2012,Teemu Selanne,ANA,46,43311.0,871.0,20.57,4.0,0.53,0.54,266.0,85.0,83.0,125667.0,8.0,12.0,6.0,10.0,36.0,7.0,126.0,63.0,85.0,468.0,497.0,16.0,31.0,12.0,4.0,7.0,8.0,5.0,89.0,3.0,35.0,35.0,161.0,145.0,116.0,166.0,8.0,51.0,49.0,0.0,24.0,10.0,0.0,3.0,84.0,119.0,169.30,132.02,2.0,871.0,167.0,96.0,23.0,131.0,72.0,209.0,1988,1,10,1,1,{'Lower body': 1},48,0.021,42.2,0.049,0,0,1,0,0,0,1,4.0,53,4500000.0,4500000.0,7.5,0,1,0,0,0,0
124,8458951,2012,Sergei Gonchar,OTT,45,64737.0,1297.0,30.12,13.0,0.56,0.54,93.0,0.0,0.0,100301.0,26.0,4.0,2.0,65.0,71.0,20.0,177.0,159.0,0.0,732.0,694.0,23.0,28.0,3.0,1.0,2.0,25.0,2.0,114.0,0.0,4.0,35.0,218.0,223.0,170.0,221.0,26.0,43.0,42.0,3.0,27.0,9.0,1.0,6.0,82.0,117.0,190.87,119.77,15.0,1297.0,191.0,85.0,10.0,120.0,74.0,210.0,1992,1,14,2,3,"{'Flu': 1, 'Lower body': 1}",48,0.062,38.5,0.073,0,1,0,0,1,0,0,8.0,102,5500000.0,5500000.0,9.2,1,1,0,0,0,0


## Step 7. Save the final dataset

We write the finished table to final_combined_dataset.csv.

In [ ]:
%%script false --no-raise-error
output_path = f"{path}/final_combined_dataset.csv"
complete.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

In [ ]:
%%script false --no-raise-error
output_path = f"{path}/final_final_complete_dataset.csv"
complete.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/SIADS 696: Milestone II/Money_Puck_Yearly_Data/final_final_complete_dataset.csv
